In [ ]:
!pip install mygene statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

import mygene
from google.colab import files

In [ ]:
uploaded = files.upload()

Saving TCGA-LUAD.star_fpkm.tsv.gz to TCGA-LUAD.star_fpkm.tsv.gz


In [ ]:
file_name = list(uploaded.keys())[0]

expression = pd.read_csv(
    file_name,
    sep="\t",
    compression="gzip"
)

expression.head()

,Ensembl_ID,TCGA-38-7271-01A,TCGA-55-7914-01A,TCGA-95-7043-01A,TCGA-73-4658-01A,TCGA-86-8076-01A,TCGA-55-7726-01A,TCGA-44-6147-01A,TCGA-50-5932-01A,TCGA-44-2661-01A,...,TCGA-50-5946-02A,TCGA-86-7713-01A,TCGA-86-8073-01A,TCGA-44-2662-01B,TCGA-MN-A4N4-01A,TCGA-53-7626-01A,TCGA-62-A46O-01A,TCGA-44-A47G-01A,TCGA-55-6969-01A,TCGA-55-6969-11A
0,ENSG00000000003.15,3.721449,4.133185,3.726275,4.518239,3.803496,3.530508,4.049657,4.388995,4.432886,...,4.376603,4.688169,3.481519,3.371141,4.180975,3.505700,5.214700,3.840785,3.262839,2.359212
1,ENSG00000000005.6,0.000000,0.000000,0.000000,2.157173,0.000000,0.000000,0.082975,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.516721,0.000000,0.000000,0.000000,0.000000,0.000000,0.035202
2,ENSG00000000419.13,4.573944,4.984061,5.348555,4.528634,4.581894,5.733986,5.059621,4.738022,5.189002,...,5.350098,4.850164,5.080670,4.915320,5.457824,4.620105,4.797184,4.304591,5.377231,4.573938
3,ENSG00000000457.14,1.874797,2.025631,1.603597,1.485375,2.021515,1.364628,2.291147,2.146688,2.018883,...,1.843783,2.477237,1.573617,2.928351,1.851199,1.935837,1.813237,1.670976,1.759326,1.514653
4,ENSG00000000460.17,1.026588,1.022900,1.059355,0.781570,0.969454,1.309409,1.453439,1.052416,1.057693,...,1.928276,2.308769,1.225460,2.924252,1.456964,1.024603,1.864057,0.916094,1.532566,0.532666


In [ ]:
expression.shape

(60660, 590)

In [ ]:
gene_column = expression.columns[0]

expression = expression.rename(
    columns={gene_column: "Ensembl_ID"}
)

expression["Ensembl_ID"] = (
    expression["Ensembl_ID"]
    .astype(str)
    .str.split(".")
    .str[0]
)

expression.head()

,Ensembl_ID,TCGA-38-7271-01A,TCGA-55-7914-01A,TCGA-95-7043-01A,TCGA-73-4658-01A,TCGA-86-8076-01A,TCGA-55-7726-01A,TCGA-44-6147-01A,TCGA-50-5932-01A,TCGA-44-2661-01A,...,TCGA-50-5946-02A,TCGA-86-7713-01A,TCGA-86-8073-01A,TCGA-44-2662-01B,TCGA-MN-A4N4-01A,TCGA-53-7626-01A,TCGA-62-A46O-01A,TCGA-44-A47G-01A,TCGA-55-6969-01A,TCGA-55-6969-11A
0,ENSG00000000003,3.721449,4.133185,3.726275,4.518239,3.803496,3.530508,4.049657,4.388995,4.432886,...,4.376603,4.688169,3.481519,3.371141,4.180975,3.505700,5.214700,3.840785,3.262839,2.359212
1,ENSG00000000005,0.000000,0.000000,0.000000,2.157173,0.000000,0.000000,0.082975,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.516721,0.000000,0.000000,0.000000,0.000000,0.000000,0.035202
2,ENSG00000000419,4.573944,4.984061,5.348555,4.528634,4.581894,5.733986,5.059621,4.738022,5.189002,...,5.350098,4.850164,5.080670,4.915320,5.457824,4.620105,4.797184,4.304591,5.377231,4.573938
3,ENSG00000000457,1.874797,2.025631,1.603597,1.485375,2.021515,1.364628,2.291147,2.146688,2.018883,...,1.843783,2.477237,1.573617,2.928351,1.851199,1.935837,1.813237,1.670976,1.759326,1.514653
4,ENSG00000000460,1.026588,1.022900,1.059355,0.781570,0.969454,1.309409,1.453439,1.052416,1.057693,...,1.928276,2.308769,1.225460,2.924252,1.456964,1.024603,1.864057,0.916094,1.532566,0.532666


In [ ]:
sample_columns = expression.columns[1:]

tumor_samples = [
    x for x in sample_columns
    if x.split("-")[3][:2] == "01"
]

normal_samples = [
    x for x in sample_columns
    if x.split("-")[3][:2] == "11"
]

print("Tumor:", len(tumor_samples))
print("Normal:", len(normal_samples))

Tumor: 528
Normal: 59


In [ ]:
results = pd.DataFrame()

results["Ensembl_ID"] = expression["Ensembl_ID"]

results["Tumor_Average"] = expression[tumor_samples].mean(axis=1)

results["Normal_Average"] = expression[normal_samples].mean(axis=1)

results["Fold_Change"] = (
    results["Tumor_Average"] + 1e-6
) / (
    results["Normal_Average"] + 1e-6
)

results["log2_Fold_Change"] = np.log2(
    results["Fold_Change"]
)

results.head()

,Ensembl_ID,Tumor_Average,Normal_Average,Fold_Change,log2_Fold_Change
0,ENSG00000000003,3.985542,2.939022,1.356077,0.439439
1,ENSG00000000005,0.141899,0.076148,1.863466,0.897988
2,ENSG00000000419,4.874947,4.551072,1.071165,0.099180
3,ENSG00000000457,1.840776,1.450156,1.269364,0.344105
4,ENSG00000000460,1.312957,0.517311,2.538040,1.343715


In [ ]:
# Force all tumor and normal expression values to numeric arrays

for col in tumor_samples + normal_samples:
    expression[col] = pd.to_numeric(
        expression[col],
        errors="coerce"
    )

# Check one tumor and one normal sample
print(expression[tumor_samples[0]].dtype)
print(expression[normal_samples[0]].dtype)

float64
float64


In [ ]:
p_values = []

for i in range(len(expression)):
    tumor = expression.loc[i, tumor_samples].astype(float).values
    normal = expression.loc[i, normal_samples].astype(float).values

    stat, p = ttest_ind(
        tumor,
        normal,
        equal_var=False
    )

    p_values.append(p)

results["p_value"] = p_values

results.head()

,Ensembl_ID,Tumor_Average,Normal_Average,Fold_Change,log2_Fold_Change,p_value
0,ENSG00000000003,3.985542,2.939022,1.356077,0.439439,9.441195e-25
1,ENSG00000000005,0.141899,0.076148,1.863466,0.897988,7.803063e-02
2,ENSG00000000419,4.874947,4.551072,1.071165,0.099180,1.827797e-11
3,ENSG00000000457,1.840776,1.450156,1.269364,0.344105,5.686428e-21
4,ENSG00000000460,1.312957,0.517311,2.538040,1.343715,8.437778e-119


In [ ]:
# Convert expression values to numeric
expression[tumor_samples + normal_samples] = (
    expression[tumor_samples + normal_samples]
    .apply(pd.to_numeric, errors="coerce")
)

# Check data types
expression[tumor_samples[:3]].dtypes

,0
TCGA-38-7271-01A,float64
TCGA-55-7914-01A,float64
TCGA-95-7043-01A,float64


In [ ]:
expression[tumor_samples + normal_samples].isna().sum().sum()

np.int64(0)

In [ ]:
i = 0

tumor = expression.loc[i, tumor_samples].astype(float).values
normal = expression.loc[i, normal_samples].astype(float).values

ttest_ind(tumor, normal, equal_var=False)

TtestResult(statistic=np.float64(14.455229197891645), pvalue=np.float64(9.44119472827043e-25), df=np.float64(86.13219015691388))

In [ ]:
from statsmodels.stats.multitest import multipletests

results["FDR"] = multipletests(
    results["p_value"],
    method="fdr_bh"
)[1]

results.head()

,Ensembl_ID,Tumor_Average,Normal_Average,Fold_Change,log2_Fold_Change,p_value,FDR
0,ENSG00000000003,3.985542,2.939022,1.356077,0.439439,9.441195e-25,NaN
1,ENSG00000000005,0.141899,0.076148,1.863466,0.897988,7.803063e-02,NaN
2,ENSG00000000419,4.874947,4.551072,1.071165,0.099180,1.827797e-11,NaN
3,ENSG00000000457,1.840776,1.450156,1.269364,0.344105,5.686428e-21,NaN
4,ENSG00000000460,1.312957,0.517311,2.538040,1.343715,8.437778e-119,NaN


In [ ]:
!pip install mygene

In [ ]:
import mygene

mg = mygene.MyGeneInfo()

mapping = mg.querymany(
    results["Ensembl_ID"].tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

gene_map = {}

for item in mapping:
    if "symbol" in item:
        gene_map[item["query"]] = item["symbol"]

results["Gene_Name"] = results["Ensembl_ID"].map(gene_map)

results.head()

INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-5000 ...
INFO:biothings.client:querying 5001-6000 ...
INFO:biothings.client:querying 6001-7000 ...
INFO:biothings.client:querying 7001-8000 ...
INFO:biothings.client:querying 8001-9000 ...
INFO:biothings.client:querying 9001-10000 ...
INFO:biothings.client:querying 10001-11000 ...
INFO:biothings.client:querying 11001-12000 ...
INFO:biothings.client:querying 12001-13000 ...
INFO:biothings.client:querying 13001-14000 ...
INFO:biothings.client:querying 14001-15000 ...
INFO:biothings.client:querying 15001-16000 ...
INFO:biothings.client:querying 16001-17000 ...
INFO:biothings.client:querying 17001-18000 ...
INFO:biothings.client:querying 18001-19000 ...
INFO:biothings.client:querying 19001-20000 ...
INFO:biothings.client:querying 20001-21000 ...
INFO:biothings.client:querying 2100

,Ensembl_ID,Tumor_Average,Normal_Average,Fold_Change,log2_Fold_Change,p_value,FDR,Gene_Name
0,ENSG00000000003,3.985542,2.939022,1.356077,0.439439,9.441195e-25,NaN,TSPAN6
1,ENSG00000000005,0.141899,0.076148,1.863466,0.897988,7.803063e-02,NaN,TNMD
2,ENSG00000000419,4.874947,4.551072,1.071165,0.099180,1.827797e-11,NaN,DPM1
3,ENSG00000000457,1.840776,1.450156,1.269364,0.344105,5.686428e-21,NaN,SCYL3
4,ENSG00000000460,1.312957,0.517311,2.538040,1.343715,8.437778e-119,NaN,FIRRM


In [ ]:
search_names = [
    "LCAL1",
    "MAGEA3",
    "PRAME",
    "AFAP1-AS1",
    "HOXB9",
    "FEZF1-AS1"
]

important_genes = results[
    results["Gene_Name"].isin(search_names)
]

important_genes[
    [
        "Gene_Name",
        "log2_Fold_Change",
        "p_value",
        "FDR",
        "Fold_Change",
        "Tumor_Average",
        "Normal_Average"
    ]
]

,Gene_Name,log2_Fold_Change,p_value,FDR,Fold_Change,Tumor_Average,Normal_Average
12882,HOXB9,4.604526,1.592631e-39,NaN,24.327659,1.055919,0.043403
16041,PRAME,5.156070,4.852632e-64,NaN,35.655932,1.620975,0.045461
24084,MAGEA3,5.454086,2.846348e-30,NaN,43.837257,1.036252,0.023638
29355,FEZF1-AS1,4.761617,1.020260e-84,NaN,27.126231,1.257093,0.046341
51863,AFAP1-AS1,4.586482,2.293761e-133,NaN,24.025289,2.202535,0.091675
58474,LCAL1,5.472497,4.377277e-84,NaN,44.400299,1.794627,0.040418


In [ ]:
results["FDR"] = multipletests(
    results["p_value"].astype(float),
    method="fdr_bh"
)[1]

important_genes = results[
    results["Gene_Name"].isin(search_names)
]

important_genes[
    [
        "Gene_Name",
        "log2_Fold_Change",
        "p_value",
        "FDR",
        "Fold_Change"
    ]
]

,Gene_Name,log2_Fold_Change,p_value,FDR,Fold_Change
12882,HOXB9,4.604526,1.592631e-39,NaN,24.327659
16041,PRAME,5.156070,4.852632e-64,NaN,35.655932
24084,MAGEA3,5.454086,2.846348e-30,NaN,43.837257
29355,FEZF1-AS1,4.761617,1.020260e-84,NaN,27.126231
51863,AFAP1-AS1,4.586482,2.293761e-133,NaN,24.025289
58474,LCAL1,5.472497,4.377277e-84,NaN,44.400299


In [ ]:
results.shape

(60660, 8)

In [ ]:
significant_genes = results[
    (results["FDR"] < 0.05) &
    (abs(results["log2_Fold_Change"]) > 1)
]

significant_genes.shape

(0, 8)

In [ ]:
results["p_value"].isna().sum(), np.isinf(results["p_value"]).sum()

(np.int64(2890), np.int64(0))

In [ ]:
from statsmodels.stats.multitest import multipletests

results_clean = results.dropna(subset=["p_value"]).copy()

results_clean = results_clean[
    np.isfinite(results_clean["p_value"])
]

results_clean["FDR"] = multipletests(
    results_clean["p_value"],
    method="fdr_bh"
)[1]

results_clean.head()

,Ensembl_ID,Tumor_Average,Normal_Average,Fold_Change,log2_Fold_Change,p_value,FDR,Gene_Name
0,ENSG00000000003,3.985542,2.939022,1.356077,0.439439,9.441195e-25,1.051104e-23,TSPAN6
1,ENSG00000000005,0.141899,0.076148,1.863466,0.897988,7.803063e-02,1.050335e-01,TNMD
2,ENSG00000000419,4.874947,4.551072,1.071165,0.099180,1.827797e-11,8.409003e-11,DPM1
3,ENSG00000000457,1.840776,1.450156,1.269364,0.344105,5.686428e-21,5.009225e-20,SCYL3
4,ENSG00000000460,1.312957,0.517311,2.538040,1.343715,8.437778e-119,9.026860e-116,FIRRM


In [ ]:
search_names = [
    "LCAL1",
    "MAGEA3",
    "PRAME",
    "AFAP1-AS1",
    "HOXB9",
    "FEZF1-AS1"
]

important_genes = results_clean[
    results_clean["Gene_Name"].isin(search_names)
]

important_genes[
    [
        "Gene_Name",
        "log2_Fold_Change",
        "p_value",
        "FDR",
        "Fold_Change"
    ]
]

,Gene_Name,log2_Fold_Change,p_value,FDR,Fold_Change
12882,HOXB9,4.604526,1.592631e-39,4.438316e-38,24.327659
16041,PRAME,5.156070,4.852632e-64,4.618395e-62,35.655932
24084,MAGEA3,5.454086,2.846348e-30,4.476818e-29,43.837257
29355,FEZF1-AS1,4.761617,1.020260e-84,2.497474e-82,27.126231
51863,AFAP1-AS1,4.586482,2.293761e-133,6.310027e-130,24.025289
58474,LCAL1,5.472497,4.377277e-84,1.044939e-81,44.400299


In [ ]:
significant_genes = results_clean[
    (results_clean["FDR"] < 0.05) &
    (abs(results_clean["log2_Fold_Change"]) > 1)
]

significant_genes.shape

(21587, 8)